# ServerWatch demo commands

Copy each cell into the right terminal. Every step is marked with where it runs:

- **💻 Local**: your computer (Git Bash on Windows).
- **☁️ EC2**: the server, after `ssh serverwatch`.
- **🌐 AWS**: the AWS console in the browser.

Replace `ELASTIC_IP` with the instance address. The talking points are in the
[presentation guide](presentation-guide.md).

---
# Part 1: Prepare your computer and launch the server

## 1. 💻 Local: open the project and check your public IP
Goes to your local copy of the project and updates it. The IP is what the security group will allow SSH from.

In [ ]:
cd PATH_TO_YOUR_LOCAL_CHECKOUT
git switch main && git pull
curl -s https://checkip.amazonaws.com

## 2. 💻 Local: create the SSH keys (G2)
Creates one key pair for the `ubuntu` admin login and one for the `junior` account. Private keys stay on this computer; only the `.pub` files are shared.

In [ ]:
mkdir -p ~/.ssh && chmod 700 ~/.ssh
ssh-keygen -t ed25519 -C serverwatch-ubuntu -f ~/.ssh/serverwatch_ec2
ssh-keygen -t ed25519 -C serverwatch-junior -f ~/.ssh/junior_key
ls -l ~/.ssh/serverwatch_ec2* ~/.ssh/junior_key*

Prints the admin public key to paste into AWS in the next step.

In [ ]:
cat ~/.ssh/serverwatch_ec2.pub

## 3. 🌐 AWS: import the key and launch the instance (G1)
- **EC2 → Key pairs → Import key pair**: paste the `.pub` text, name it `serverwatch`.
- **Launch instance**: Ubuntu 24.04 LTS, a free-tier type, key pair `serverwatch`.
- **Security group**: SSH 22 from **My IP** only (HTTP and HTTPS come later).
- **Elastic IPs → Allocate**, then **Associate** it with the instance.

## 4. 💻 Local: SSH config with short names
Saves the address, user and key per server, so `ssh serverwatch` replaces the long command. `IdentitiesOnly` makes SSH offer only that key.

In [ ]:
IP=ELASTIC_IP
cat >> ~/.ssh/config <<EOF

Host serverwatch
    HostName $IP
    User ubuntu
    IdentityFile ~/.ssh/serverwatch_ec2
    IdentitiesOnly yes

Host serverwatch-junior
    HostName $IP
    User junior
    IdentityFile ~/.ssh/junior_key
    IdentitiesOnly yes
EOF
chmod 600 ~/.ssh/config
cat ~/.ssh/config

For a **new instance** (after a rehearsal), don't append again: change `HostName` in `nano ~/.ssh/config`, and forget the old server's fingerprint:

In [ ]:
ssh-keygen -R OLD_ELASTIC_IP

## 5. 💻 Local: first connection (G2)
Logs in as `ubuntu`. Answer `yes` to the fingerprint question; SSH then remembers this server and warns if it ever changes.

In [ ]:
ssh serverwatch

## 6. 💻 Local: copy the setup script with SCP (G3)
Sends `setup.sh` to the server's home folder over the same encrypted SSH connection. Run in a second local terminal, in the project folder.

In [ ]:
scp setup.sh serverwatch:~/setup.sh

---
# Part 2: Set up the server

## 7. ☁️ EC2: clone the repository (G4)
Installs Git and clones the project into the `ubuntu` user's home, so that user owns the code and no service needs root.

In [ ]:
sudo apt-get update && sudo apt-get install -y git
git clone https://github.com/EmirU116/server-api.git ~/serverwatch
cd ~/serverwatch && git log -1 --oneline

## 8. ☁️ EC2: save the Discord webhook (G6)
Stores the webhook in a file only you can read; the prompt hides what you paste.

In [ ]:
umask 077
install -d -m 700 "$HOME/.config/serverwatch-alerts"
if IFS= read -rsp 'Discord webhook URL: ' webhook; then
    printf '%s\n' "$webhook" > "$HOME/.config/serverwatch-alerts/discord-webhook"
fi
unset webhook
printf '\n'

## 9. ☁️ EC2: run setup (G3)
Installs everything: PostgreSQL, the API and collector services, Nginx, Cron backups, Docker, the `senior` and `junior` accounts, the Discord timer and the `sw-*` aliases.

In [ ]:
export SERVERWATCH_SENIOR_USER=senior
export SERVERWATCH_JUNIOR_USER=junior
export SERVERWATCH_DISCORD_WEBHOOK_FILE="$HOME/.config/serverwatch-alerts/discord-webhook"
SERVERWATCH_PROJECT_DIR="$HOME/serverwatch" bash ~/setup.sh ELASTIC_IP

Reloads the shell so the new aliases work, then lists them.

In [ ]:
source ~/.bashrc
sw-help

## 10. Second account with its own key (G2)
💻 **Local:** print the junior public key.

In [ ]:
cat ~/.ssh/junior_key.pub

☁️ **EC2:** create the account's `.ssh` folder and paste the public key in nano.

In [ ]:
sudo install -d -o junior -g junior -m 700 /home/junior/.ssh
sudo -u junior nano /home/junior/.ssh/authorized_keys
sudo chmod 600 /home/junior/.ssh/authorized_keys

💻 **Local:** log in as `junior` with its own key and show its groups (it can write to dev, only read prod).

In [ ]:
ssh serverwatch-junior
id
exit

---
# Part 3: Show the G requirements

## 11. ☁️ EC2: verify the deployment (G7, G9)
Read-only checks of the services, database, collector and Nginx; expect 8 passed, 0 failed.

In [ ]:
cd ~/serverwatch
bash verify.sh ELASTIC_IP

## 12. ☁️ EC2: Nginx by hand (G8)
Opens the reverse-proxy config in nano, checks the syntax, and reloads without dropping connections.

In [ ]:
cd /etc/nginx/sites-enabled
ls -l
sudo nano serverwatch.conf
sudo nginx -t
sudo systemctl reload nginx
cd ~/serverwatch

## 13. 💻 Local: public access (G7, G8)
🌐 First add **HTTP 80 from anywhere** to the security group. These requests come from your computer, through Nginx, to the API; then open `http://ELASTIC_IP/docs` in the browser.

In [ ]:
curl -s "http://ELASTIC_IP/api/v1/health?server_id=1"
curl -s -o /dev/null -w '%{http_code}\n' http://ELASTIC_IP/docs

## 14. ☁️ EC2: systemd and the database (G7, G9)
Shows the API and collector running under systemd, and that the API and PostgreSQL only listen on 127.0.0.1.

In [ ]:
sw-status
systemctl cat serverwatch-api
sudo ss -ltnp | grep -E ':5432|:8000'

## 15. ☁️ EC2: Cron backup (G5)
Shows the daily 03:15 schedule. In `sudoedit`, set it to the next minute; put `15 3` back after it has run.

In [ ]:
sudo cat /etc/cron.d/serverwatch-backup
sudoedit /etc/cron.d/serverwatch-backup

Shows that Cron ran the backup and lists the snapshots.

In [ ]:
sudo journalctl -t serverwatch-backup -n 10 --no-pager
sudo ls -lt /var/backups/serverwatch

Restores the newest snapshot into a separate database and counts the rows: a backup only counts if it can be restored.

In [ ]:
(
    set -euo pipefail
    unset PGHOST PGHOSTADDR PGPORT PGSERVICE PGSERVICEFILE
    snapshot=/var/backups/serverwatch/$(sudo ls -1 /var/backups/serverwatch | tail -n 1)
    restore_db="serverwatch_restore_$(date +%s)"
    sudo -u postgres createdb --host=/var/run/postgresql --port=5432 "$restore_db"
    sudo cat "$snapshot/database.dump" |
        sudo -u postgres pg_restore --host=/var/run/postgresql --port=5432 \
            --exit-on-error --no-owner --no-privileges --dbname="$restore_db"
    sudo -u postgres psql -X --host=/var/run/postgresql --port=5432 \
        --set=ON_ERROR_STOP=1 --dbname="$restore_db" \
        -c 'SELECT count(*) AS metrics, max(collected_at) AS latest FROM metrics' \
        -c 'SELECT count(*) AS incidents FROM incidents'
    printf 'Restored database: %s\n' "$restore_db"
)

## 16. Optional: copy a backup home (G5)
☁️ **EC2:** packs the newest snapshot into a file you can download (the backup folder is root-only).

In [ ]:
latest=$(sudo ls -1 /var/backups/serverwatch | tail -n 1)
sudo tar -C /var/backups/serverwatch -czf ~/serverwatch-backup.tgz "$latest"
sudo chown "$USER": ~/serverwatch-backup.tgz

💻 **Local:** downloads it and removes the copy on the server. Your computer pulls the file because its IP changes, so the server can't reliably send to it.

In [ ]:
mkdir -p ~/serverwatch-backups
scp serverwatch:~/serverwatch-backup.tgz ~/serverwatch-backups/
ssh serverwatch 'rm ~/serverwatch-backup.tgz'
ls -l ~/serverwatch-backups

## 17. ☁️ EC2: Discord alert (G6)
Lowers the disk threshold below current usage, so the timer posts to Discord within a minute.

In [ ]:
cd ~/serverwatch
sw-timers
df -h /
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 25

After the message arrives, put the normal 90% threshold back.

In [ ]:
bash infrastructure/bootstrap/setup_discord_alerts.sh "$HOME/.config/serverwatch-alerts/discord-webhook" 90

---
# Part 4: Show the VG requirements

## 18. ☁️ EC2: give `ubuntu` access to Docker (VG1)
Adds `ubuntu` to the `docker` group so it can run Compose without sudo. Group changes apply at the next login, so reconnect.

In [ ]:
sudo usermod -aG docker "$USER"
exit

💻 **Local:** reconnect, then check that Docker answers without sudo.

In [ ]:
ssh serverwatch
docker info --format '{{.ServerVersion}}'

## 18b. ☁️ EC2: Docker Compose (VG1)
Builds the images and starts db → migrate → api → web; `migrate` showing `Exited (0)` is success.

In [ ]:
cd ~/serverwatch
bash scripts/deployment/compose_host.sh start
bash scripts/deployment/compose_host.sh ps -a

Sends this host's metrics to the Compose API; stop with Ctrl+C after about 20 seconds.

In [ ]:
bash scripts/deployment/compose_host.sh collect

## 19. ☁️ EC2: data survives container recreation (VG1)
Counts the rows, removes the containers (the volume stays), starts them again and counts again; both counts must match.

In [ ]:
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"
bash scripts/deployment/compose_host.sh down
bash scripts/deployment/compose_host.sh up -d --wait
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"

## 20. Domain and HTTPS (VG2)
🌐 In Namecheap → **Advanced DNS**, point the `@` and `www` A records at the Elastic IP; in AWS add **HTTPS 443 from anywhere**.

💻 **Local:** asks Cloudflare's DNS directly, so your router's cache can't show an old address.

In [ ]:
nslookup serverwatch.dev 1.1.1.1

☁️ **EC2:** gets a Let's Encrypt certificate with Certbot, turns on HTTPS and checks renewal. For rehearsals, put `SERVERWATCH_CERTBOT_STAGING=1` in front to save real certificates.

In [ ]:
cd ~/serverwatch
bash infrastructure/bootstrap/setup_https.sh serverwatch.dev ELASTIC_IP

☁️ **EC2:** shows the certificate, the renewal timer and the HTTPS Nginx site.

In [ ]:
sudo certbot certificates
systemctl list-timers certbot.timer
sudo nano /etc/nginx/sites-enabled/serverwatch-domain.conf

💻 **Local:** HTTPS answers with a valid certificate, and HTTP redirects to it (301). Then open `https://serverwatch.dev`.

In [ ]:
curl -s "https://serverwatch.dev/api/v1/health?server_id=1"
curl -s -o /dev/null -w '%{http_code} %{redirect_url}\n' http://serverwatch.dev/

## 21. Optional: deploy after CI (VG4 bonus)
Needs the setup in [continuous deployment](continuous-deployment.md). 💻 **Local:** push a small change, then watch CI and *ServerWatch deploy* under **Actions** on GitHub.

In [ ]:
git commit -am "Demo change"
git push

☁️ **EC2:** shows which commit GitHub deployed and the images tagged with it.

In [ ]:
cat ~/.config/serverwatch-delivery/deployed-commit
bash ~/serverwatch/scripts/deployment/compose_host.sh ps

---
## 22. 🌐 Afterwards
Watch the recording and check the link. Then in AWS: **stop or terminate the instance** and **release the Elastic IP** (an unattached Elastic IP costs money).